# Demo data

**What it does**
Runs one complete interpretation with the fake AI: stores two small sample PDFs as a new case, runs the three agents (summaries and test steps), and saves the result as a case interpretation with its test steps in the database. Every run adds one more case.

**Why**
So `06_database.ipynb` and `05_test_steps.ipynb` have records to show, without real documents, without cloud access and without cost. It goes through exactly the same services as the API's `POST /api/v1/test-steps`; only the AI answers come from `tests/fixtures/fake_ai/`.

**How to use**
1. Set `KCT_DB_DRIVER=sqlite` in the project's `.env` (or in the first code cell). With `memory` the records would disappear when the kernel stops.
2. Run all cells (Run All).
3. Then open `06_database.ipynb` and run it: the new case interpretation and its 17 test steps are there.

The AI provider is forced to `fake` in this notebook only, whatever `.env` says.

## Setup

Finds the project folder, reads `.env`, and switches this notebook to the fake AI.

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from services.config import load_env_file, settings

load_env_file()
os.environ["KCT_AI_PROVIDER"] = "fake"   # demo answers, no cloud, no cost
# os.environ["KCT_DB_DRIVER"] = "sqlite"  # uncomment if .env does not set it

config = settings()
print("AI provider:", config.ai_provider)
print("Database:   ", config.db_driver, config.db_path if config.db_driver == "sqlite" else "")
if config.db_driver == "memory":
    print("⚠️ memory: the demo records exist only while this kernel runs. Set KCT_DB_DRIVER=sqlite to keep them.")

## Run the interpretation on two sample PDFs

Writes two tiny placeholder PDFs, stores them as a new case (`storage/uploads/case-NNN/`), runs the three fake agents and saves the result. Takes a second.

In [ ]:
from services.app import boot

kct = boot()

sample = settings().storage_root / "demo"
sample.mkdir(parents=True, exist_ok=True)
pdf = b"%PDF-1.4\n% demo document\n%%EOF\n"
(sample / "control_details_demo.pdf").write_bytes(pdf)
(sample / "test_plan_demo.pdf").write_bytes(pdf)

interpretation = kct.interpretations.create_and_generate(
    [sample / "control_details_demo.pdf"], [sample / "test_plan_demo.pdf"]
)
meta = interpretation.metadata
print("✅ Saved case interpretation", meta.case_interpretation_id)
print("   case:", meta.case_reference, "| steps:", meta.step_count, "| database:", settings().db_driver)

## What was saved

All case interpretations in the database, newest first. The same list `06_database.ipynb` shows.

In [ ]:
for m in kct.database.interpretations():
    print(m.case_interpretation_id, m.case_reference, f"{m.step_count:>3} steps", m.created_at[:19])